# Data Validation and Preparation

This notebook first **validates** your forecast card CSV files (`project-*.csv`, `poi-*.csv`, `observations-*.csv`, `forecast-*.csv`) against the forecastcards schema by checking for wrong/missing columns, missing required values, invalid values, and time-range errors. Then, walks you through **filling in missing data** interactively. Every accepted answer is written straight back to the original CSV file, so progress is saved as you go and is never lost if you stop partway through.

**What counts as "missing" for the fill-in step:** a cell is flagged if it is blank, or if it contains (case-insensitive) `None`, `unknown`, `n/a`, `na`, or `null`.

**How to use this notebook:**
1. Set `DATA_FOLDER` to the folder containing all your project CSVs (all projects' files can sit in one folder together — the script tells files apart by their filename prefix: `project-`, `poi-`, `observations-`, `forecast-`).
2. Run the "validate" cell to check structure, required fields, invalid values, and time errors for every project. Review `validation_report.txt` (saved in `DATA_FOLDER`) for the full, untruncated results. Fix any structural issues directly in the CSVs before continuing.
3. Run the "scan" cell to see a summary of what's missing.
4. Run the "fix" cell to go through missing values one at a time. At each prompt you can:
   - type a value to fill it in
   - type `skip` to leave it for later (it'll show up again next time you scan)
   - type `unknown` if that's a genuine, confirmed answer
   - type `quit` to stop the session early (anything fixed so far is already saved)
5. Run the "re-scan" cell (and optionally re-validate) to confirm everything is filled in.

In [ ]:
import importlib
import os
import pandas as pd
import forecastcards
importlib.reload(forecastcards)
from forecastcards import missing_data_tools as mdt
importlib.reload(mdt)

## If output error occurs (ModuleNotFoundError: No module named 'forecastcards'), try running the following command in your terminal:
## pip install -e

## 1. Set your data folder

Point this at the folder holding all your CSVs (project, poi, observations, and forecast).

In [ ]:
# Update this path to point at your folder of CSVs
DATA_FOLDER = r"C:\Users\hmod225\Documents\Code\forecastcards_ba_data\data"

import os
print(os.path.exists(DATA_FOLDER))
print(os.listdir(DATA_FOLDER))

## 2. Validate structure and required fields

This checks every `project-*.csv` (and its linked `poi`, `observations`, `forecast` cards) against the forecastcards schema, and reports, per project:
- **Structure** — required columns that are completely missing, or columns present that aren't in the schema (typos/renames).
- **Missing data** — required fields that are blank or a placeholder value.
- **Invalid values** — values not in an allowed list, or the wrong type (e.g. text where a number/date/time is expected).
- **Time errors** — rows where `start_time` isn't before `end_time`.

It prints a summary plus full per-project detail for **every** project (passed and failed), and also saves the complete report to `validation_report.txt` inside `DATA_FOLDER`.

**Fix structure issues directly in the CSV file.** Missing *data* (blank/placeholder values in otherwise-correct columns) can instead be filled in with the interactive tool in step 4 below.

In [ ]:
results = forecastcards.validate_data_folder(DATA_FOLDER)

## 3. Scan for missing data

This reads every recognized CSV in `DATA_FOLDER` and reports how many missing values were found, broken down by file and column.

In [ ]:
scan_results = mdt.scan_folder(DATA_FOLDER)
mdt.print_scan_summary(scan_results)

## 4. Fix missing data interactively

Running this cell starts the prompt-and-fill session described above. It processes every flagged file found in `DATA_FOLDER`.

**Tip:** if you only want to work through one project or one file type at a time, pass `file_filter` with a substring that appears in those filenames, e.g. `file_filter="40974"` for one project's files, or `file_filter="project-"` for just project files across all projects.

In [ ]:
# Fix ALL missing values across every file in DATA_FOLDER
mdt.fix_missing_interactive(DATA_FOLDER)

### Optional: Fix just one project or file type at a time

Uncomment and adjust below if you'd rather work through projects one at a time.

In [ ]:
# mdt.fix_missing_interactive(DATA_FOLDER, file_filter="MN_F-M-0436")   # just one project's files
# mdt.fix_missing_interactive(DATA_FOLDER, file_filter="project-")  # just project-*.csv files across all projects

## 5. Re-scan to confirm everything is filled in

Run the scan again (it always reflects the current state of the files on disk, since fixes are written immediately). You can also re-run the validate cell above to confirm structure/values are still clean.

In [ ]:
scan_results = mdt.scan_folder(DATA_FOLDER)
mdt.print_scan_summary(scan_results)